# Two-Tower Training

**Goal:** Train the two-tower retrieval model end-to-end with MLflow logging and checkpoint saving.

**Sections:**
1. Imports & config
2. Data preparation
3. DataLoaders
4. Model + optimizer
5. MLflow setup
6. Training loop
7. Validation

## 1. Imports & config

In [1]:
import sys
sys.path.insert(0, "..")

import random
from pathlib import Path

import mlflow
import numpy as np
import torch
import torch.nn as nn
from omegaconf import OmegaConf
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from src.data.loaders import prepare_data
from src.data.sampler import (
    TwoTowerCollator,
    build_confirmed_neg_index,
    make_weighted_sampler,
)
from src.models.two_tower import TwoTowerModel
from src.train.losses import infonce_loss

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

/home/s38976581_gmail_com/micromamba/envs/fin_sentiment/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch: 2.5.1+cu121
cuda available: True
device: NVIDIA L4


In [2]:
# Load config from baseline.yaml
cfg = OmegaConf.load("../configs/baseline.yaml")
print(OmegaConf.to_yaml(cfg))

data:
  train_path: gs://nshen7-personal-bucket/projects/rec_sys_goodreads/data/two_tower_splits/train
  val_path: gs://nshen7-personal-bucket/projects/rec_sys_goodreads/data/two_tower_splits/val
  confirmed_negatives_path: gs://nshen7-personal-bucket/projects/rec_sys_goodreads/data/two_tower_splits/confirmed_negatives
  artifacts_path: artifacts/vocab_and_stats.pt
  mmap_dir: artifacts/mmap
  num_workers: 4
  prefetch_factor: 2
vocab:
  num_users: null
  num_items: null
  num_authors: null
  num_languages: null
  num_formats: null
  num_shelves: null
model:
  d_id: 128
  d_cat: 32
  d_out: 256
  normalize: true
  user_mlp_hidden:
  - 512
  - 256
  item_mlp_hidden:
  - 512
  - 256
  dropout: 0.1
history:
  length: 10
  pad_item_id: 0
numeric:
  book_avg_rating:
    use_log1p: false
    declared_min: 0.0
    declared_max: 5.0
  book_ratings_count:
    use_log1p: true
    declared_min: null
    declared_max: null
  book_num_pages:
    use_log1p: true
    declared_min: null
    declared_m

In [3]:
# Reproducibility
SEED = cfg.training.seed
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

Using device: cuda


## 2. Data preparation

- First run: reads train Parquet from GCS, builds vocab/normalization artifacts, saves to `artifacts/vocab_and_stats.pt`
- Subsequent runs: loads artifacts from cache (fast)

In [4]:
# Change working dir to project root so relative paths (artifacts/, checkpoints/) resolve correctly
import os
os.chdir("..")
print("Working directory:", os.getcwd())

Working directory: /home/s38976581_gmail_com/projects/rec_sys_goodreads/two_tower


In [5]:
train_dataset, val_dataset, item_cat_feats, item_num_feats, artifacts = prepare_data(cfg)

Loading artifacts from artifacts/vocab_and_stats.pt
  num_users=876,134  num_items=2,260,810
  author vocab size: 170,567
  language vocab size: 116
  format vocab size: 702
  shelf vocab size: 27,299
Building item feature tensors (chunked)...


  item_cat_feats: (2260811, 6)  torch.int64
  item_num_feats: (2260811, 5)  torch.float32
Wrapping train dataset...
  Loading memory-mapped arrays from artifacts/mmap/train
Wrapping val dataset...
  Loading memory-mapped arrays from artifacts/mmap/val
  train: 29,683,247 rows  |  val: 8,450,964 rows


In [6]:
# Sanity-check a single sample
sample = train_dataset[0]
for k, v in sample.items():
    print(f"{k:30s}  shape={tuple(v.shape) if hasattr(v, 'shape') else '()'}  dtype={v.dtype}")
    ## print the first few values for each tensor
    if isinstance(v, torch.Tensor):
        print(f"  values: {v.flatten()[:5].tolist()}")
    ## Numeric features should be normalized (min=0, max=1)

user_id                         shape=()  dtype=torch.int64
  values: [216427]
target_item_id                  shape=()  dtype=torch.int64
  values: [1580]
history_item_ids                shape=(10,)  dtype=torch.int64
  values: [1004, 1494, 334, 18119, 14648]
history_item_weights            shape=(10,)  dtype=torch.float32
  values: [4.0, 1.0, 1.0, 1.0, 1.0]
sample_weight                   shape=()  dtype=torch.float32
  values: [1.0]
target_cat_feats                shape=(6,)  dtype=torch.int64
  values: [60804, 27, 696, 22619, 10232]
target_num_feats                shape=(5,)  dtype=torch.float32
  values: [0.7599999904632568, 0.5536686778068542, 0.4232165217399597, 0.949999988079071, 0.0]


/home/s38976581_gmail_com/projects/rec_sys_goodreads/two_tower/notebooks/../src/data/loaders.py:508: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at ../torch/csrc/utils/tensor_numpy.cpp:206.)
  "history_item_ids": torch.from_numpy(self.history_item_ids[idx]).long(),


In [7]:
!free -h

               total        used        free      shared  buff/cache   available
Mem:            15Gi       5.5Gi       6.1Gi       148Mi       4.0Gi       9.6Gi
Swap:          8.0Gi       2.4Gi       5.6Gi


## 3. DataLoaders

In [8]:
# Confirmed negatives index — loaded once, shared across all batches
print("Loading confirmed negatives index...")
conf_neg_index = build_confirmed_neg_index(cfg.data.confirmed_negatives_path)
print(f"  {len(conf_neg_index):,} users with confirmed negatives")

Loading confirmed negatives index...


  22,576 users with confirmed negatives


In [9]:
!free -h

               total        used        free      shared  buff/cache   available
Mem:            15Gi       5.6Gi       6.1Gi       148Mi       4.0Gi       9.6Gi
Swap:          8.0Gi       2.4Gi       5.6Gi


In [10]:
collator = TwoTowerCollator(
    conf_neg_index=conf_neg_index,
    item_cat_feats=item_cat_feats,
    item_num_feats=item_num_feats,
    cfg=cfg,
)

# Weighted sampler: higher-strength interactions seen more often
train_sampler = make_weighted_sampler(
    sample_weights=train_dataset.sample_weights,
    transform=cfg.training.sample_weight_transform,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.training.batch_size,
    sampler=train_sampler,
    collate_fn=collator,
    num_workers=cfg.data.num_workers,
    prefetch_factor=cfg.data.prefetch_factor,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=cfg.training.batch_size,
    shuffle=False,
    collate_fn=collator,
    num_workers=cfg.data.num_workers,
    prefetch_factor=cfg.data.prefetch_factor,
    pin_memory=True,
)

print(f"Train batches per epoch: {len(train_loader):,}")
print(f"Val   batches per epoch: {len(val_loader):,}")

Train batches per epoch: 14,494
Val   batches per epoch: 4,127


In [11]:
# Inspect one batch
batch = next(iter(train_loader))
print("Batch keys and shapes:")
for k, v in batch.items():
    print(f"  {k:30s}  {str(tuple(v.shape)):20s}  {v.dtype}")

Batch keys and shapes:
  user_id                         (2048,)               torch.int64
  target_item_id                  (2048,)               torch.int64
  history_item_ids                (2048, 10)            torch.int64
  history_item_weights            (2048, 10)            torch.float32
  sample_weight                   (2048,)               torch.float32
  item_ids                        (2304,)               torch.int64
  item_cat_feats                  (2304, 6)             torch.int64
  item_num_feats                  (2304, 5)             torch.float32
  item_is_positive                (2304,)               torch.bool


## 4. Model + optimizer

In [12]:
model = TwoTowerModel(cfg, artifacts).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(model)

Total parameters:     408,550,016
Trainable parameters: 408,550,016
TwoTowerModel(
  (user_id_embedding): IDEmbedding(
    (embedding): Embedding(876135, 128)
  )
  (item_id_embedding): IDEmbedding(
    (embedding): Embedding(2260811, 128, padding_idx=0)
  )
  (author_embedding): CategoricalEmbedding(
    (embedding): Embedding(170567, 32, padding_idx=0)
  )
  (language_embedding): CategoricalEmbedding(
    (embedding): Embedding(116, 32, padding_idx=0)
  )
  (format_embedding): CategoricalEmbedding(
    (embedding): Embedding(702, 32, padding_idx=0)
  )
  (shelf_embedding): ShelfEmbedding(
    (embedding): Embedding(27299, 32, padding_idx=0)
  )
  (item_tower): ItemTower(
    (mlp): Sequential(
      (0): Linear(in_features=261, out_features=512, bias=True)
      (1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (2): ReLU()
      (3): Dropout(p=0.1, inplace=False)
      (4): Linear(in_features=512, out_features=256, bias=True)
      (5): LayerNorm((256,), eps=1e-05, ele

In [13]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=cfg.training.lr,
    weight_decay=cfg.training.weight_decay,
)
print(optimizer)

Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 1e-05
)


In [14]:
# Verify one forward pass before committing to training
model.train()
batch_gpu = {k: v.to(DEVICE) for k, v in batch.items()}

user_vecs, item_vecs = model(batch_gpu)
loss = infonce_loss(
    user_vecs, item_vecs,
    batch_gpu["item_is_positive"],
    temperature=cfg.training.temperature,
)

print(f"user_vecs : {tuple(user_vecs.shape)}")
print(f"item_vecs : {tuple(item_vecs.shape)}")
print(f"loss      : {loss.item():.4f}  (expected ~log({cfg.training.batch_size}) = {np.log(cfg.training.batch_size):.2f} at random init)")

user_vecs : (2048, 256)
item_vecs : (2304, 256)
loss      : 7.7442  (expected ~log(2048) = 7.62 at random init)


## 5. MLflow setup

In [15]:
## includ date and time in the experiment name to avoid overwriting previous runs
import datetime
now = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
MLFLOW_TRACKING_URI = "mlruns"   # local file-based tracking under two_tower/mlruns/
EXPERIMENT_NAME = f"two_tower_baseline_{now}"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)
print(f"MLflow tracking URI : {mlflow.get_tracking_uri()}")
print(f"Experiment           : {EXPERIMENT_NAME}")
print("Run `mlflow ui` in the two_tower/ directory to open the UI")

/home/s38976581_gmail_com/micromamba/envs/fin_sentiment/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/02/28 03:36:14 INFO mlflow.tracking.fluent: Experiment with name 'two_tower_baseline_2026-02-28_03-36-14' does not exist. Creating a new experiment.


MLflow tracking URI : mlruns
Experiment           : two_tower_baseline_2026-02-28_03-36-14
Run `mlflow ui` in the two_tower/ directory to open the UI


## 6. Training loop

In [16]:
MAX_TRAIN_STEPS = 1000  # set to None to run full epoch
MAX_VAL_STEPS   = 200   # set to None to run full validation

def train_one_epoch(model, loader, optimizer, cfg, device, global_step, mlflow_run, epoch):
    """Train for one epoch. Returns (avg_loss, global_step)."""
    model.train()
    total_loss = 0.0
    log_loss = 0.0
    log_every = cfg.training.log_every_n_steps
    n_batches = len(loader)

    pbar = tqdm(loader, desc=f"Epoch {epoch} train", leave=True, total=MAX_TRAIN_STEPS or n_batches)
    for step, batch in enumerate(pbar):
        if MAX_TRAIN_STEPS and step >= MAX_TRAIN_STEPS:
            break

        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}

        optimizer.zero_grad()
        user_vecs, item_vecs = model(batch)
        loss = infonce_loss(
            user_vecs, item_vecs,
            batch["item_is_positive"],
            temperature=cfg.training.temperature,
        )
        loss.backward()
        optimizer.step()

        loss_val = loss.item()
        total_loss += loss_val
        log_loss += loss_val
        global_step += 1

        if global_step % log_every == 0:
            avg = log_loss / log_every
            mlflow.log_metric("train/loss_step", avg, step=global_step)
            pbar.set_postfix({"step_loss": f"{avg:.4f}", "step": global_step})
            print(f"  [epoch {epoch} | step {global_step} | batch {step+1}/{n_batches}]  loss={avg:.4f}")
            log_loss = 0.0

    n_steps = min(step + 1, MAX_TRAIN_STEPS or n_batches)
    avg_epoch_loss = total_loss / n_steps
    return avg_epoch_loss, global_step

In [17]:
def validate(model, loader, cfg, device):
    """Run validation. Returns avg loss."""
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for step, batch in enumerate(tqdm(loader, desc="val", leave=False, total=MAX_VAL_STEPS or len(loader))):
            if MAX_VAL_STEPS and step >= MAX_VAL_STEPS:
                break
            batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
            user_vecs, item_vecs = model(batch)
            loss = infonce_loss(
                user_vecs, item_vecs,
                batch["item_is_positive"],
                temperature=cfg.training.temperature,
            )
            total_loss += loss.item()

    n_steps = min(step + 1, MAX_VAL_STEPS or len(loader))
    return total_loss / n_steps

In [ ]:
CHECKPOINT_DIR = Path(cfg.training.checkpoint_dir) / f"example_{EXPERIMENT_NAME}"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

def save_checkpoint(model, optimizer, epoch, val_loss, run_id):
    path = CHECKPOINT_DIR / f"epoch_{epoch:02d}_val{val_loss:.4f}.pt"
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "val_loss": val_loss,
        "mlflow_run_id": run_id,
    }, path)
    print(f"  Checkpoint saved: {path}")
    return path

In [19]:
import time

# ── Main training loop ──────────────────────────────────────────────────────

with mlflow.start_run() as run:
    run_id = run.info.run_id
    print(f"MLflow run ID: {run_id}")

    # Log all config params flat
    flat_cfg = OmegaConf.to_container(cfg, resolve=True)
    def _flatten(d, prefix=""):
        out = {}
        for k, v in d.items():
            key = f"{prefix}{k}"
            if isinstance(v, dict):
                out.update(_flatten(v, prefix=key + "."))
            else:
                out[key] = v
        return out
    mlflow.log_params(_flatten(flat_cfg))
    mlflow.log_param("num_users", artifacts["num_users"])
    mlflow.log_param("num_items", artifacts["num_items"])
    mlflow.log_param("total_params", total_params)

    global_step = 0
    best_val_loss = float("inf")
    train_start = time.time()

    for epoch in range(1, cfg.training.epochs + 1):
        print(f"\n{'='*60}")
        print(f"Epoch {epoch}/{cfg.training.epochs}  (global step so far: {global_step:,})")
        print(f"{'='*60}")

        epoch_start = time.time()
        train_loss, global_step = train_one_epoch(
            model, train_loader, optimizer, cfg, DEVICE, global_step, run, epoch
        )
        train_elapsed = time.time() - epoch_start
        print(f"\n  Train loss: {train_loss:.4f}  ({train_elapsed:.0f}s)")
        mlflow.log_metric("train/loss_epoch", train_loss, step=epoch)

        val_start = time.time()
        val_loss = validate(model, val_loader, cfg, DEVICE)
        val_elapsed = time.time() - val_start
        print(f"  Val   loss: {val_loss:.4f}  ({val_elapsed:.0f}s)")
        mlflow.log_metric("val/loss_epoch", val_loss, step=epoch)

        total_elapsed = time.time() - train_start
        print(f"  Total elapsed: {total_elapsed/60:.1f} min")

        # Save checkpoint every epoch; track best
        ckpt_path = save_checkpoint(model, optimizer, epoch, val_loss, run_id)
        mlflow.log_artifact(str(ckpt_path), artifact_path="checkpoints")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            mlflow.log_metric("val/best_loss", best_val_loss, step=epoch)
            print(f"  ** New best val loss: {best_val_loss:.4f}")

print(f"\nTraining complete. Best val loss: {best_val_loss:.4f}")

MLflow run ID: f49c05cc2cb24aad94b0cfcbca174619

Epoch 1/2  (global step so far: 0)


Epoch 1 train:  10%|█         | 101/1000 [01:27<02:41,  5.56it/s, step_loss=7.5408, step=100]

  [epoch 1 | step 100 | batch 100/14494]  loss=7.5408


Epoch 1 train:  20%|██        | 201/1000 [01:45<02:23,  5.57it/s, step_loss=7.1310, step=200]

  [epoch 1 | step 200 | batch 200/14494]  loss=7.1310


...

Epoch 2 train: 100%|██████████| 1000/1000 [04:17<00:00,  5.57it/s, step_loss=6.7006, step=2000]

  [epoch 2 | step 2000 | batch 1000/14494]  loss=6.7006


Epoch 2 train: 100%|██████████| 1000/1000 [04:18<00:00,  3.87it/s, step_loss=6.7006, step=2000]



  Train loss: 6.7228  (259s)


  Val   loss: 7.0048  (13s)
  Total elapsed: 9.6 min
  Checkpoint saved: checkpoints/two_tower_baseline_2026-02-28_03-36-14/epoch_02_val7.0048.pt
  ** New best val loss: 7.0048

Training complete. Best val loss: 7.0048


## 7. Quick sanity check on trained model

In [20]:
# Check that user/item vectors are unit-norm (because normalize=True in config)
model.eval()
with torch.no_grad():
    b = next(iter(val_loader))
    b = {k: v.to(DEVICE) for k, v in b.items()}
    u, it = model(b)
    print(f"user_vecs norm (mean): {u.norm(dim=-1).mean():.4f}  (expect ~1.0)")
    print(f"item_vecs norm (mean): {it.norm(dim=-1).mean():.4f}  (expect ~1.0)")

    # Score distribution: positives (diagonal) vs negatives (off-diagonal)
    scores = u @ it.T   # [B, B+N_neg]
    B = u.size(0)
    pos_scores = scores[torch.arange(B), torch.arange(B)]
    # mask off-diagonal for neg scores
    mask = torch.ones_like(scores, dtype=torch.bool)
    mask[torch.arange(B), torch.arange(B)] = False
    neg_scores = scores[mask]

    print(f"Positive score mean : {pos_scores.mean():.4f}")
    print(f"Negative score mean : {neg_scores.mean():.4f}")
    print(f"Score gap (pos-neg) : {pos_scores.mean() - neg_scores.mean():.4f}  (should be > 0)")

user_vecs norm (mean): 1.0000  (expect ~1.0)
item_vecs norm (mean): 1.0000  (expect ~1.0)
Positive score mean : 0.5486
Negative score mean : 0.0711
Score gap (pos-neg) : 0.4774  (should be > 0)
